# Домашнее задание № 2. Мешок слов

## Задание 1 (3 балла)

У векторайзеров в sklearn есть встроенная токенизация на регулярных выражениях. Найдите способо заменить её на кастомную токенизацию

Обучите векторайзер с дефолтной токенизацией и с токенизацией razdel.tokenize. Обучите классификатор (любой) с каждым из векторизаторов. Сравните метрики и выберете победителя.

(в вашей тетрадке должен быть код обучения и все метрики; если вы сдаете в .py файлах то сохраните полученные метрики в отдельном файле или в комментариях)

In [5]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity

In [22]:
!pip install razdel

In [6]:
!unzip data/labeled.csv.zip -d data/

unzip:  cannot find or open data/labeled.csv.zip, data/labeled.csv.zip.zip or data/labeled.csv.zip.ZIP.


In [8]:
data = pd.read_csv('labeled.csv')
data.head()


,comment,toxic
0,"Верблюдов-то за что? Дебилы, бл...\n",1.0
1,"Хохлы, это отдушина затюканого россиянина, мол...",1.0
2,Собаке - собачья смерть\n,1.0
3,"Страницу обнови, дебил. Это тоже не оскорблени...",1.0
4,"тебя не убедил 6-страничный пдф в том, что Скр...",1.0


In [10]:
train, test = train_test_split(data, test_size=0.1, shuffle=True)
train.reset_index(inplace=True)
test.reset_index(inplace=True)

In [ ]:
# Векторайзер с дефолтной токенизацией - vectorizer_count_default
vectorizer_count_default = CountVectorizer(
    max_features=10000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 3)
)
X = vectorizer_count_default.fit_transform(train.comment)
X_test = vectorizer_count_default.transform(test.comment)
y = train.toxic.values
y_test = test.toxic.values

clf = LogisticRegression(C=0.1, class_weight='balanced')
clf.fit(X, y)
preds = clf.predict(X_test)

print(classification_report(y_test, preds, zero_division=0))

              precision    recall  f1-score   support

         0.0       0.91      0.82      0.87       977
         1.0       0.69      0.83      0.76       465

    accuracy                           0.83      1442
   macro avg       0.80      0.83      0.81      1442
weighted avg       0.84      0.83      0.83      1442



In [ ]:
# vectorizer_tfid_default
vectorizer_tfid_default = TfidfVectorizer(
    max_features=10000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 3)
)
X = vectorizer_tfid_default.fit_transform(train.comment)
X_test = vectorizer_tfid_default.transform(test.comment)
y = train.toxic.values
y_test = test.toxic.values

clf = LogisticRegression(C=0.1, class_weight='balanced')
clf.fit(X, y)
preds = clf.predict(X_test)

print(classification_report(y_test, preds, zero_division=0))

              precision    recall  f1-score   support

         0.0       0.88      0.87      0.88       977
         1.0       0.73      0.75      0.74       465

    accuracy                           0.83      1442
   macro avg       0.81      0.81      0.81      1442
weighted avg       0.83      0.83      0.83      1442



In [115]:
from razdel import tokenize
import re

def clean_text(text):
  text = re.sub(r'\[^\w\s\.\,\!\?\-\:\(\)\"\'\`\&]', '', text)
  text = re.sub(r'\s+', ' ', text)
  return text.strip()

def token_razdel(text):
  clean = clean_text(text)
  tokens = list(tokenize(text))
  filtered_tokens = []
  for token in tokens:
        word = token.text.lower().strip()
        if (len(word) > 2 and
            not re.match(r'^[\d\W_]+$', word) and
            word not in russian_stop_words):
            filtered_tokens.append(word)

  return filtered_tokens

In [116]:
# Векторайзер с токенизацией razdel - vectorizer_tfid_custom

vectorizer_tfid_custom = TfidfVectorizer(
    max_features=10000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 3),
    tokenizer=token_razdel,
    token_pattern=None
)
X = vectorizer_tfid_custom.fit_transform(train.comment)
X_test = vectorizer_tfid_custom.transform(test.comment)
y = train.toxic.values
y_test = test.toxic.values

clf = LogisticRegression(C=0.1, class_weight='balanced')
clf.fit(X, y)
preds = clf.predict(X_test)

print(classification_report(y_test, preds, zero_division=0))

              precision    recall  f1-score   support

         0.0       0.89      0.85      0.87       985
         1.0       0.70      0.77      0.74       457

    accuracy                           0.82      1442
   macro avg       0.80      0.81      0.80      1442
weighted avg       0.83      0.82      0.83      1442



In [117]:
# vectorizer_count_custom
vectorizer_count_custom = CountVectorizer(
    max_features=10000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 3),
    tokenizer=token_razdel,
    token_pattern=None
)

X = vectorizer_count_custom.fit_transform(train.comment)
X_test = vectorizer_count_custom.transform(test.comment)
y = train.toxic.values
y_test = test.toxic.values

clf = LogisticRegression(C=0.1, class_weight='balanced')
clf.fit(X, y)
preds = clf.predict(X_test)

print(classification_report(y_test, preds, zero_division=0))

              precision    recall  f1-score   support

         0.0       0.92      0.78      0.84       985
         1.0       0.64      0.86      0.73       457

    accuracy                           0.80      1442
   macro avg       0.78      0.82      0.79      1442
weighted avg       0.83      0.80      0.81      1442



Лучше всех справился **vectorizer_tfid_default** (tfid с дефолтной токенизацией), метрики более сбалансированны и стабильны, меньше всех ложных предсказаний (precision 0.69 для 1.0)



---
Кастомная токенизация не показала улучшений, а только ухудшила результаты для обоих векторайзеров (особенно для **vectorizer_count_custom**, он часто ошибается - precision 0.64 для 1.0)


---




## Задание 2 (3 балла)

Обучите 2 любых разных классификатора из семинара. Предскажите токсичность для текстов из тестовой выборки (используйте одну и ту же выборку для обоих классификаторов) и найдите 10 самых токсичных для каждого из классификаторов. Сравните получаемые тексты - какие тексты совпадают, какие отличаются, правда ли тексты токсичные?

Требования к моделям:   
а) один классификатор должен использовать CountVectorizer, другой TfidfVectorizer  
б) у векторазера должны быть вручную заданы как минимум 5 параметров (можно ставить разные параметры tfidfvectorizer и countvectorizer)  
в) у классификатора должно быть задано вручную как минимум 2 параметра (по возможности)  
г)  f1 мера каждого из классификаторов должна быть минимум 0.75  

*random_seed не считается за параметр

In [ ]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
russian_stop_words = stopwords.words('russian')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [13]:
# CountVectorizer + MultinomialNB

vectorizer_count = CountVectorizer(
    max_features=11496,
    min_df=2,
    max_df=0.7,
    ngram_range=(1, 2),
    stop_words=russian_stop_words,
    binary=True
)
X_count = vectorizer_count.fit_transform(train.comment)
X_test_count = vectorizer_count.transform(test.comment)
y_count = train.toxic.values
y_test_count = test.toxic.values

clf_nb = MultinomialNB(alpha=1.0, fit_prior=True)
clf_nb.fit(X_count, y_count)
preds_nb = clf_nb.predict(X_test_count)
probs_nb = clf_nb.predict_proba(X_test_count)[:, 1]

print(classification_report(y_test_count, preds_nb))

              precision    recall  f1-score   support

         0.0       0.87      0.91      0.89       985
         1.0       0.78      0.72      0.75       457

    accuracy                           0.85      1442
   macro avg       0.83      0.81      0.82      1442
weighted avg       0.84      0.85      0.84      1442



In [16]:
# TfidVectorizer + KNeighborsClassifier

vectorizer_tfid = TfidfVectorizer(
    max_features=12000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 3),
    stop_words=None
)
X_tfid = vectorizer_tfid.fit_transform(train.comment)
X_test_tfid = vectorizer_tfid.transform(test.comment)
y_tfid = train.toxic.values
y_test_tfid = test.toxic.values

clf_knn = KNeighborsClassifier(
    n_neighbors=15,
    metric='cosine',
    weights='distance'
)
clf_knn.fit(X_tfid, y_tfid)
preds_knn = clf_knn.predict(X_test_tfid)
probs_knn = clf_knn.predict_proba(X_test_tfid)[:, 1]

print(classification_report(y_test_tfid, preds_knn))

              precision    recall  f1-score   support

         0.0       0.86      0.83      0.84       985
         1.0       0.65      0.70      0.68       457

    accuracy                           0.79      1442
   macro avg       0.76      0.76      0.76      1442
weighted avg       0.79      0.79      0.79      1442



In [17]:
results = pd.DataFrame({
    'text': test.comment.values,
    'prob_toxic_nb': probs_nb,
    'prob_toxic_knn': probs_knn,
    'prediction_nb': (preds_knn >= 0.5).astype(int),
    'prediction_knn': (preds_nb >= 0.5).astype(int)
})

In [18]:
top_toxic_nb = results.nlargest(10, 'prob_toxic_nb')[['text', 'prob_toxic_nb']]
top_toxic_knn = results.nlargest(10, 'prob_toxic_knn')[['text', 'prob_toxic_knn']]

In [19]:
top_10_nb = top_toxic_nb.nlargest(10, 'prob_toxic_nb')[['text', 'prob_toxic_nb']]
top_10_nb

,text,prob_toxic_nb
298,С каких пор порноскримеры нарушают что-то В то...,1.0
454,"самый сброд червей-пидоров Не, ну если это чер...",1.0
666,Ну давай разберём всё тобой написанное. Бляядь...,1.0
829,"лахтадырые и ольгинцы (Лахта, Ольгино) это кот...",1.0
958,А сейчас смотрит хуйню всякую с пидорасом звон...,1.0
998,"В role reversal треде fet поселился шизик, пор...",1.0
1237,"та ну, хуйня это все про джентельменство . мен...",1.0
1172,Я русский и мне бомбит от шария. Потому что он...,1.0
713,Да тупая баба. Видос недавно был: мразь какая ...,1.0
1031,"Не уйду, хуй соси мой ты. Я вообще из рашки и ...",1.0


from matplotlib import pyplot as plt
top_10_nb['prob_toxic_nb'].plot(kind='hist', bins=20, title='prob_toxic_nb')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
top_10_nb['prob_toxic_nb'].plot(kind='line', figsize=(8, 4), title='prob_toxic_nb')
plt.gca().spines[['top', 'right']].set_visible(False)

In [20]:
top_10_knn = top_toxic_knn.nlargest(10, 'prob_toxic_knn')[['text', 'prob_toxic_knn']]
top_10_knn

,text,prob_toxic_knn
29,Что блять за конференция мамашек?\n,1.0
59,На мешке пылесоса я скачууу\n,1.0
135,А вот и мамкины аметисты набежали.\n,1.0
214,Иди на хуй козлина ебучая езё мне указывать бу...,1.0
244,"Пыня здесь не надолго, скоро в очередной раз п...",1.0
323,Эти дизлайки подпаленных грязноштанников\n,1.0
346,"Хуй чурки с причмоком проглотит, Пизду и со сп...",1.0
372,Гроб гроб кладбище пидор\n,1.0
466,"Что ты имеешь против кремля, холоп заморского ...",1.0
476,Главное сохраняет морду лица,1.0


Все тексты различаются, но в обоих вариантах все тексты токсичные.

## Задание 3 (4 балла - 1 балл за каждый классификатор)

Для классификаторов Logistic Regression, Decision Trees, Naive Bayes, RandomForest найдите способ извлечь важность признаков для предсказания токсичного класса. Сопоставьте полученные числа со словами (или нграммами) в словаре и найдите топ - 5 "токсичных" слов для каждого из классификаторов.

Важное требование: в топе не должно быть стоп-слов. Для этого вам нужно будет правильным образом настроить векторизацию.
Также как и в предыдущем задании у классификаторов должно быть задано вручную как минимум 2 параметра (по возможности, f1 мера каждого из классификаторов должна быть минимум 0.75

In [128]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
russian_stop_words = ['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впрочем', 'хорошо', 'свою', 'этой', 'перед', 'иногда', 'лучше', 'чуть', 'том', 'нельзя', 'такой', 'им', 'более', 'всегда', 'конечно', 'всю', 'между', 'тебе', 'очень']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [129]:
# Naive Bayes
vectorizer_naive = CountVectorizer(
    max_features=11496,
    min_df=2,
    max_df=0.7,
    ngram_range=(1, 3),
    stop_words=russian_stop_words,
    binary=True,
    tokenizer=token_razdel,
    token_pattern=None
)
X_naive = vectorizer_naive.fit_transform(train.comment)
X_test_naive = vectorizer_naive.transform(test.comment)
y_naive = train.toxic.values
y_test_naive = test.toxic.values

nb = MultinomialNB(alpha=1.0, fit_prior=False)
nb.fit(X_naive, y_naive)
preds_naive = nb.predict(X_test_naive)

print(classification_report(y_test_naive, preds_naive))

              precision    recall  f1-score   support

         0.0       0.89      0.87      0.88       985
         1.0       0.73      0.76      0.75       457

    accuracy                           0.84      1442
   macro avg       0.81      0.82      0.81      1442
weighted avg       0.84      0.84      0.84      1442



In [130]:
# LogisticRegression
vectorizer_logis = CountVectorizer(
    max_features=15000,
    min_df=2,
    max_df=0.7,
    ngram_range=(1, 2),
    stop_words=russian_stop_words,
    tokenizer=token_razdel,
    token_pattern=None
)
X_logis = vectorizer_logis.fit_transform(train.comment)
X_test_logis = vectorizer_logis.transform(test.comment)
y_logis = train.toxic.values
y_test_logis = test.toxic.values

lr = LogisticRegression(C=1.0, class_weight='balanced')
lr.fit(X_logis, y_logis)
preds_logis = lr.predict(X_test_logis)

print(classification_report(y_test_logis, preds_logis))

              precision    recall  f1-score   support

         0.0       0.91      0.83      0.86       985
         1.0       0.68      0.82      0.74       457

    accuracy                           0.82      1442
   macro avg       0.80      0.82      0.80      1442
weighted avg       0.84      0.82      0.83      1442



In [131]:
# DecisionTreeClassifier
vectorizer_destree = CountVectorizer(
    max_features=12000,
    min_df=2,
    max_df=0.7,
    ngram_range=(1, 3),
    stop_words=russian_stop_words,
    binary=True,
    tokenizer=token_razdel,
    token_pattern=None
)
X_destree = vectorizer_destree.fit_transform(train.comment)
X_test_destree = vectorizer_destree.transform(test.comment)
y_destree = train.toxic.values
y_test_destree = test.toxic.values

dt = DecisionTreeClassifier(max_depth=15,class_weight='balanced')
dt.fit(X_destree, y_destree)
preds_destree = dt.predict(X_test_destree)

print(classification_report(y_test_destree, preds_destree))

              precision    recall  f1-score   support

         0.0       0.83      0.36      0.50       985
         1.0       0.38      0.85      0.52       457

    accuracy                           0.51      1442
   macro avg       0.61      0.60      0.51      1442
weighted avg       0.69      0.51      0.51      1442



In [132]:
# RandomForestClassifier
vectorizer_random = CountVectorizer(
    max_features=12000,
    min_df=2,
    max_df=0.7,
    ngram_range=(1, 3),
    stop_words=russian_stop_words
)
X_random = vectorizer_random.fit_transform(train.comment)
X_test_random = vectorizer_random.transform(test.comment)
y_random = train.toxic.values
y_test_random = test.toxic.values

rf = RandomForestClassifier(max_depth=15,class_weight='balanced')
rf.fit(X_random, y_random)
preds_random = rf.predict(X_test_random)

print(classification_report(y_test_random, preds_random))

              precision    recall  f1-score   support

         0.0       0.88      0.67      0.76       985
         1.0       0.53      0.80      0.64       457

    accuracy                           0.71      1442
   macro avg       0.70      0.74      0.70      1442
weighted avg       0.77      0.71      0.72      1442



In [32]:
! pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 48.1 MB/s eta 0:00:00


In [133]:
feature_names_nb = vectorizer_naive.get_feature_names_out()
feature_names_lr = vectorizer_logis.get_feature_names_out()
feature_names_dt = vectorizer_destree.get_feature_names_out()
feature_names_rf = vectorizer_random.get_feature_names_out()


In [134]:
import pymorphy3

def get_top_toxic_features(clf, feature_names, model_name, top_n=5):
    morph = pymorphy3.MorphAnalyzer()

    if hasattr(clf, 'coef_'):  # Logistic Regression
        importance_scores = clf.coef_[0]
    elif hasattr(clf, 'feature_importances_'):  # Decision Trees, Random Forest
        importance_scores = clf.feature_importances_
    elif hasattr(clf, 'feature_log_prob_'):  # Naive Bayes
        importance_scores = clf.feature_log_prob_[1] - clf.feature_log_prob_[0]

    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': importance_scores,
        'normal_form': [morph.parse(word)[0].normal_form for word in feature_names]
    })

    grouped = feature_importance.groupby('normal_form').agg({
        'feature': 'first',
        'importance': 'max'
    }).reset_index(drop=True)

    top_features = grouped.nlargest(top_n, 'importance')
    return top_features[['feature', 'importance']]

top_nb = get_top_toxic_features(nb, feature_names_nb, "Naive Bayes")
top_lr = get_top_toxic_features(lr, feature_names_lr, "Logistic Regression")
top_dt = get_top_toxic_features(dt, feature_names_dt, "Decision Tree")
top_rf = get_top_toxic_features(rf, feature_names_rf, "Random Forest")


In [135]:
top_nb

,feature,importance
5040,сука,4.953223
5606,хохлов,4.665541
963,дебил,4.606352
5609,хохла,4.448128
4235,русню,4.326767


In [136]:
top_lr

,feature,importance
7491,хохлов,3.074117
7499,хохла,2.949944
1271,дебил,2.617567
7050,тупа,2.238692
6675,сука,2.041060


In [137]:
top_dt

,feature,importance
5890,хохла,0.082468
2693,нахуй,0.070000
5887,хохлов,0.063597
2642,например,0.060566
1545,знаем,0.048386


In [138]:
top_rf

,feature,importance
6120,хохлов,0.021867
1112,гг,0.017669
6124,хохла,0.017320
567,блядь,0.016040
3017,нахуй,0.015931
